In [1]:
import pandas as pd
from configparser import ConfigParser

# Read in the configurations from the params.ini file
config = ConfigParser()
config.read('params.ini')

def read_file(filename, dataset):

    # Read the file
    with open(filename, 'r') as file:
        lines = file.readlines()

    data = []
    for line in lines:
        parts = line.strip().split(' ')
        parts = list(filter(lambda x: x.strip(), parts)) # Filter the values if there are empty spaces present
        label = int(parts[0])
        features = [0] * int(config[dataset]['n_features']) # Initialise the features list according to the number of n_features
        for part in parts[1:]:
            index, value = part.split(':')
            features[int(index) - 1] = float(value)  # Subtracting 1 to adjust 1-based to 0-based indexing

        data.append([label] + features) # Add the label and feature in a line

    # Create column names for DataFrame
    columns = ['label'] + [f'feature_{i}' for i in range(1, int(config[dataset]['n_features'])+1)]

    # Create a DataFrame from the list of lists and return it
    return pd.DataFrame(data, columns=columns)


if __name__ == '__main__':
    # Read each file and assign the returned df
    dna_train = read_file(config['DNA']['train'], 'DNA')
    dna_test = read_file(config['DNA']['test'], 'DNA')
    dna_val = read_file(config['DNA']['val'], 'DNA')

    splice_train = read_file(config['SPLICE']['train'], 'SPLICE')
    splice_test = read_file(config['SPLICE']['test'], 'SPLICE')

    protein_train = read_file(config['PROTEIN']['train'], 'PROTEIN')
    protein_test = read_file(config['PROTEIN']['test'], 'PROTEIN')
    protein_val = read_file(config['PROTEIN']['val'], 'PROTEIN')

In [2]:
protein_val.describe()

,label,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_348,feature_349,feature_350,feature_351,feature_352,feature_353,feature_354,feature_355,feature_356,feature_357
count,2871.000000,2871.000000,2871.000000,2871.000000,2871.000000,2871.000000,2871.000000,2871.000000,2871.000000,2871.000000,...,2871.000000,2871.000000,2871.000000,2871.000000,2871.000000,2871.000000,2871.000000,2871.000000,2871.000000,2871.000000
mean,0.717868,0.070606,0.074427,0.049512,0.015963,0.034368,0.015138,0.034612,0.079073,0.075545,...,0.058732,0.021439,0.021470,0.041773,0.050408,0.033532,0.056256,0.046308,0.053734,0.052943
std,0.822055,0.172279,0.193297,0.142528,0.077984,0.134225,0.101621,0.140042,0.223991,0.177340,...,0.153317,0.124276,0.102067,0.141469,0.138142,0.111734,0.152355,0.138493,0.154467,0.223959
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,0.040000,0.030000,0.020000,0.000000,0.000000,0.000000,0.000000,0.020000,0.060000,...,0.040000,0.000000,0.000000,0.010000,0.030000,0.010000,0.030000,0.030000,0.020000,0.000000
max,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [3]:
protein_val.head()

,label,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_348,feature_349,feature_350,feature_351,feature_352,feature_353,feature_354,feature_355,feature_356,feature_357
0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.23,0.0,0.03,0.00,0.02,0.06,0.04,0.27,0.14,0.0
1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.08,0.0,0.01,0.05,0.31,0.10,0.03,0.18,0.02,0.0
2,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.0,0.01,0.00,0.00,0.00,0.00,0.00,0.00,0.0
3,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.10,0.0,0.06,0.04,0.05,0.01,0.07,0.05,0.04,0.0
4,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.07,0.0,0.01,0.02,0.10,0.13,0.06,0.02,0.05,0.0


In [6]:
# data_train = protein_train
# data_test = protein_test
# data_val = protein_val

data_train = dna_train
data_test = dna_test
data_val = dna_val

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Define your neural network architecture
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

# Define your dataset class
class ProteinDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Assuming you have your dataset split into train, test, and validation sets
protein_train_dataset = ProteinDataset(data_train, data_train['label'])
protein_test_dataset = ProteinDataset(data_test, data_test['label'])
protein_val_dataset = ProteinDataset(data_val, data_val['label'])

# Hyperparameters
input_size = protein_train.shape[1]
hidden_size = 128
num_classes = len(protein_train['label'].unique())
batch_size = 64
learning_rate = 0.001
num_epochs = 10

# Create DataLoader for each dataset
train_loader = DataLoader(dataset=protein_train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(dataset=protein_test_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(dataset=protein_val_dataset, batch_size=batch_size, shuffle=False)

# Initialize the model, loss function, and optimizer
model = SimpleNN(input_size, hidden_size, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training the model
# total_step = len(train_loader)
# for epoch in range(num_epochs):
#     for i, (inputs, labels) in enumerate(train_loader):
#         # Forward pass
#         outputs = model(inputs)
#         loss = criterion(outputs, labels)

#         # Backward and optimize
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         if (i+1) % 100 == 0:
#             print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'
#                    .format(epoch+1, num_epochs, i+1, total_step, loss.item()))

total_step = len(train_loader)
print("Total steps per epoch:", total_step)
for epoch in range(num_epochs):
    for i, (inputs, labels) in enumerate(train_loader):
        try:
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'
                    .format(epoch+1, num_epochs, i+1, total_step, loss.item()))
        except KeyError as e:
            print("KeyError occurred at epoch {}, step {}: {}".format(epoch+1, i+1, e))


# Evaluation on the validation set
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for inputs, labels in val_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Accuracy on the validation set: {} %'.format(100 * correct / total))

# Evaluation on the test set
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Accuracy on the test set: {} %'.format(100 * correct / total))


Total steps per epoch: 21


KeyError: 1386

In [ ]:
print("Length of training dataset:", len(protein_train_dataset))
total_step = len(train_loader)
print("Number of steps per epoch:", total_step)

Length of training dataset: 14895
Number of steps per epoch: 232


In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

In [ ]:
len(protein_train['label'].unique())

3

In [ ]:
# Define hyperparameters
input_size =  protein_train.shape[1]
hidden_size = 128
num_classes =  len(protein_train['label'].unique())
learning_rate = 0.001
num_epochs = 10
batch_size = 1

# Prepare your data loaders
train_loader = DataLoader(protein_train, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(protein_val, batch_size=batch_size)
test_loader = DataLoader(protein_test, batch_size=batch_size)

# Initialize the model
model = SimpleClassifier(input_size, hidden_size, num_classes)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Initialize the model
model = SimpleClassifier(input_size, hidden_size, num_classes)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Training the model
for epoch in range(num_epochs):
    model.train()
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    
    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {loss.item():.4f}, Validation Loss: {val_loss/len(val_loader):.4f}, Validation Accuracy: {(correct/total)*100:.2f}%")

KeyError: 12107

In [ ]:
print(protein_train.shape)
print(protein_test.shape)
print(protein_val.shape)

(14895, 358)
(6621, 358)
(2871, 358)
